## **Chapter 3. Looking Inside Large Language Models**

In [ ]:
!pip install transformers torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# Carregando um modelo leve para testes rápidos no Colab
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Modelo carregado com sucesso!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo carregado com sucesso!


**Exercício 01 - O que é Token? (A base do loop)**

**Contexto**: Antes de entendermos o loop que gera palavras, precisamos ver o que o modelo realmente enxerga. Ele não lê letras, ele lê "tokens" (que são números representando palavras ou pedaços de palavras).

**Questão**: Use o `tokenizer` para transformar a frase "A inteligência artificial" em números (IDs). Depois, use a função de decodificar para transformar o segundo número de volta em texto.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

frase = "A inteligência artificial"

# Transforma o texto em números (IDs)
tokens = tokenizer.encode(frase)
print("Tokens em números:", tokens)

# Pegue o SEGUNDO número da lista e decodifique:
# DICA: use tokenizer.decode([ ____ ])
token_decodificado = tokenizer.decode([ 5832 ])
print("O segundo número significa a palavra:", token_decodificado)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Tokens em números: [319, 13856, 335, 10544, 23116]
O segundo número significa a palavra: asing


**Exercício 02 - Inspecionando a "Linha de Montagem" do Transformer**

**Contexto**:
Um modelo *Transformer* não é uma caixa preta mágica; ele funciona como uma linha de montagem industrial. Para entender o que acontece lá dentro, precisamos olhar para os dois "portões" principais do modelo:

* **O Portão de Entrada (Embeddings)**: Onde os IDs numéricos das palavras são convertidos em vetores matemáticos (o significado abstrato).

* **O Portão de Saída (LM Head)**: Onde o "raciocínio" final do modelo é traduzido de volta para probabilidades de palavras do nosso vocabulário.

**Questão**:
Utilize os métodos oficiais da biblioteca `transformers` para inspecionar a estrutura dessas duas camadas. Embora cada modelo possa ter nomes internos diferentes (como embed_tokens ou wte), os métodos `get_input_embeddings()` e `get_output_embeddings()` funcionam de forma universal.

Complete o código abaixo para exibir as configurações técnicas da entrada e da saída do modelo carregado:

In [ ]:
# O modelo já foi carregado anteriormente como 'model' (1° célula)

# Acesse e imprima a camada de entrada (Embeddings):
print("Estrutura da Camada de Entrada (Embeddings)")
# SEU CÓDIGO AQUI:
print(model.get_input_embeddings())

# Acesse e imprima a camada de saída (LM Head):
print("\nEstrutura da Camada de Saída (LM Head)")
# SEU CÓDIGO AQUI:
print(model.get_output_embeddings())

Estrutura da Camada de Entrada (Embeddings)
Embedding(50257, 768)

Estrutura da Camada de Saída (LM Head)
Linear(in_features=768, out_features=50257, bias=False)


**Exercício 03 - Atenção (Q, K, V) - A Analogia do Banco de Dados**

**Contexto**: Em vez de usar matemática complexa, vamos usar a analogia do texto. A Query (Q) é o que você pesquisa. A Key (K) é a etiqueta da informação. O Value (V) é o conteúdo real.

**Questão**:
Crie um dicionário simples que atue como o nosso "Banco de Dados" (Keys e Values). Depois, crie uma Query para buscar o significado da palavra "banco".

In [ ]:
# Nossas Keys e Values
memoria_do_modelo = {
    "madeira": "Material duro que vem das árvores.",
    "banco": "Assento longo para várias pessoas sentarem.",
    "dinheiro": "Moedas e notas usadas para pagamento."
}

# Sua Query (O que o modelo está tentando entender agora)
query = "dinheiro"

# Busque a 'query' dentro da 'memoria_do_modelo' e imprima o 'Value'
resultado = memoria_do_modelo[query]

print(f"Para a query '{query}', o valor encontrado foi: {resultado}")

Para a query 'dinheiro', o valor encontrado foi: Moedas e notas usadas para pagamento.


**Exercício 04 - O Impacto da Temperatura no Sampling**

**Contexto**: Após calcular as probabilidades do próximo token, o modelo precisa escolher um. A estratégia Greedy sempre pega o mais provável (bom para fatos). A estratégia de Sampling (amostragem) sorteia o token com base nas probabilidades, e o parâmetro `temperature` controla o nível de "loucura" ou criatividade desse sorteio.

**Questão**: Gere um texto a partir do prompt "O maior desafio da inteligência artificial no futuro será" usando amostragem (`do_sample=True`). Teste gerar o texto duas vezes: uma com `temperature=0.1` (muito conservador) e outra com `temperature=2.0` (muito caótico). Imprima os dois resultados e observe a diferença.

In [ ]:
prompt_teste = "O maior desafio dos sistemas operacionais no futuro será"
inputs_teste = tokenizer(prompt_teste, return_tensors="pt")

# Gere com Temperatura 0.1
output_baixa = model.generate(
    **inputs_teste,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.1
)

# Gere com Temperatura 2.0
output_alta = model.generate(
    **inputs_teste,
    max_new_tokens=30,
    do_sample=True,
    temperature=2.0
)

# Imprimindo os resultados:
print("Temperatura 0.1 (Conservadora)")
print(tokenizer.decode(output_baixa[0], skip_special_tokens=True))

print("\nTemperatura 2.0 (Caótica)")
print(tokenizer.decode(output_alta[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Temperatura 0.1 (Conservadora)
O maior desafio da inteligência artificial no futuro seráomin Que min artificial hast	  futuro seráomin Que min artificial hast	  futuro seráom

Temperatura 2.0 (Caótica)
O maior desafio da inteligência artificial no futuro seráal futiceamp dur($ wVerallasnone Пре), como	lack
ст play te ш virtualgroup cant mperty An m=" on


**Exercício 05 - Medindo o Poder do KV Cache**

**Contexto**: O KV Cache salva as matrizes de Key e Value das palavras anteriores na memória. Sem isso, o modelo teria que recalcular a atenção de toda a frase a cada nova palavra gerada, tornando o processo cada vez mais lento.

**Questão**: Crie um benchmark simples. Gere um texto de 30 tokens (`max_new_tokens=30`) a partir de um prompt qualquer. Faça isso duas vezes medindo o tempo com a biblioteca `time`: a primeira passando o argumento `use_cache=False` no `model.generate` e a segunda com `use_cache=True`. Imprima a diferença de tempo.

In [ ]:
prompt_cache = "Para construir um modelo de linguagem eficiente, precisamos de"
inputs_cache = tokenizer(prompt_cache, return_tensors="pt")

# Teste SEM cache:
inicio = time.time()
output_sem = model.generate(
    **inputs_cache,
    max_new_tokens=30,
    use_cache=False,
    do_sample=False
)
tempo_sem_cache = time.time() - inicio

# Teste COM cache:
inicio = time.time()
output_com = model.generate(
    **inputs_cache,
    max_new_tokens=30,
    use_cache=True,
    do_sample=False
)
tempo_com_cache = time.time() - inicio

print(f"Tempo SEM cache: {tempo_sem_cache:.2f}s")
print(f"Tempo COM cache: {tempo_com_cache:.2f}s")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Tempo SEM cache: 11.83s
Tempo COM cache: 3.00s
